# Step 1: Preprocess Image-Caption Pairs (for AdaBelief training)

Extract features from (image, caption) pairs:
- Caption -> text model embeddings [max_len, dim]
- Image -> image model features [dim]

Supported text models:
- `phobert-base` -> [768] (PhoBERT, general Vietnamese)
- `phobert-base-v2` -> [768] (PhoBERT v2, 7x more training data)
- `visobert` -> [768] (ViSoBERT, social media specialist)
- `phobert-large` -> [1024] (PhoBERT large)

Supported image models:
- `resnet50` -> [2048] (ImageNet baseline)
- `clip-vit-L-14` -> [1024] (CLIP, pre-aligned with text)
- `siglip-base-224` -> [768] (SigLIP, multilingual)

- Input: `../workflow_coolant/pairs/pairs_{train,dev,test}.json`
- Output: `./processed/coolant_<image>_<text>_<split>.h5`

In [6]:
import sys
import os
import gc
import numpy as np
import torch
import h5py
from pathlib import Path
from tqdm import tqdm

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [ ]:
# Config — change image_model / text_model to switch feature extractors
CONFIG = {
    "text_model": "visobert",      # Options: "phobert-base" [768], "phobert-base-v2" [768], "visobert" [768], "phobert-large" [1024]
    "image_model": "clip-vit-L-14~",          # Options: "resnet50" [2048], "clip-vit-L-14" [1024], "siglip-base-224" [768]
    "language": "vi",
    "max_length": 128,  # Captions are short (not full articles)
    "batch_size": 32,
}

# Derive short names for filenames
_IMAGE_SHORT = CONFIG["image_model"].replace("/", "_").replace("-", "_")
_TEXT_SHORT = CONFIG["text_model"].replace("/", "_").replace("-", "_")
FILE_PREFIX = f"coolant_{_IMAGE_SHORT}_{_TEXT_SHORT}"

# Read pairs from workflow_coolant, output to local processed/
PAIRS_DIR = Path("../workflow_coolant/pairs")
OUTPUT_DIR = Path("./processed")
OUTPUT_DIR.mkdir(exist_ok=True)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")
print(f"Text model:  {CONFIG['text_model']}")
print(f"Image model: {CONFIG['image_model']}")
print(f"File prefix: {FILE_PREFIX}")
print(f"  e.g. {FILE_PREFIX}_train.h5")

Device: cuda
Text model:  visobert
Image model: clip-vit-L-14
File prefix: coolant_clip_vit_L_14_visobert
  e.g. coolant_clip_vit_L_14_visobert_train.h5


In [8]:
# Initialize preprocessor
from src.preprocessing import CombinedPreprocessor

preprocessor = CombinedPreprocessor(
    text_model_name=CONFIG["text_model"],
    image_model_name=CONFIG["image_model"],
    language=CONFIG["language"],
    max_text_length=CONFIG["max_length"],
    device=device,
)
print("Preprocessor ready")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

c:\Users\tungh\.conda\envs\fake_news\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tungh\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias         

Preprocessor ready


In [9]:
# Load pairs and preprocess each split
from src.processing.coolant import PairExtractor

BATCH_SIZE = CONFIG["batch_size"]

for split in ["train", "dev", "test"]:
    pairs_path = PAIRS_DIR / f"pairs_{split}.json"
    if not pairs_path.exists():
        print(f"{split}: pairs file not found, skipping")
        continue

    pairs = PairExtractor.load_pairs(str(pairs_path))
    captions = [p["caption"] for p in pairs]
    image_paths = [p["image_path"] for p in pairs]
    article_ids = [p["article_idx"] for p in pairs]
    total = len(pairs)
    n_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\nProcessing {split}: {total} pairs, {n_batches} batches")

    all_caption_feat, all_image_feat = [], []

    for i in tqdm(range(n_batches), desc=split):
        s, e = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, total)

        cap_feat = preprocessor.text_preprocessor.extract_token_embeddings(captions[s:e])
        img_feat = preprocessor.image_preprocessor.extract_features(image_paths[s:e])

        all_caption_feat.append(cap_feat)
        all_image_feat.append(img_feat)

        del cap_feat, img_feat
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    caption_features = np.vstack(all_caption_feat)
    image_features = np.vstack(all_image_feat)
    article_id_array = np.array(article_ids)

    # Save HDF5 with model names in filename
    hdf5_path = OUTPUT_DIR / f"{FILE_PREFIX}_{split}.h5"
    chunk_size = min(64, total)

    with h5py.File(hdf5_path, "w") as f:
        f.create_dataset(
            "caption_features", data=caption_features,
            chunks=(chunk_size, caption_features.shape[1], caption_features.shape[2]),
            compression="gzip", compression_opts=4,
        )
        f.create_dataset(
            "image_features", data=image_features,
            chunks=(chunk_size, image_features.shape[1]),
            compression="gzip", compression_opts=4,
        )
        f.create_dataset("article_ids", data=article_id_array)
        f.attrs["n_samples"] = total
        f.attrs["caption_shape"] = caption_features.shape[1:]
        f.attrs["image_shape"] = image_features.shape[1:]
        f.attrs["image_model"] = CONFIG["image_model"]
        f.attrs["image_dim"] = image_features.shape[1]
        f.attrs["text_model"] = CONFIG["text_model"]
        f.attrs["text_dim"] = caption_features.shape[2]

    size_mb = hdf5_path.stat().st_size / (1024 * 1024)
    print(f"  caption={caption_features.shape}, image={image_features.shape}")
    print(f"  Saved: {hdf5_path} ({size_mb:.1f} MB)")

    del all_caption_feat, all_image_feat, caption_features, image_features
    gc.collect()

print("\nAll splits preprocessed.")


Processing train: 3537 pairs, 111 batches


train: 100%|██████████| 111/111 [03:22<00:00,  1.83s/it]


  caption=(3537, 128, 768), image=(3537, 1024)
  Saved: processed\coolant_clip_vit_L_14_visobert_train.h5 (1245.2 MB)

Processing dev: 1054 pairs, 33 batches


dev: 100%|██████████| 33/33 [01:01<00:00,  1.86s/it]


  caption=(1054, 128, 768), image=(1054, 1024)
  Saved: processed\coolant_clip_vit_L_14_visobert_dev.h5 (371.1 MB)

Processing test: 876 pairs, 28 batches


test: 100%|██████████| 28/28 [00:45<00:00,  1.61s/it]


  caption=(876, 128, 768), image=(876, 1024)
  Saved: processed\coolant_clip_vit_L_14_visobert_test.h5 (308.4 MB)

All splits preprocessed.


In [10]:
# Verify HDF5 files
for split in ["train", "dev", "test"]:
    path = OUTPUT_DIR / f"{FILE_PREFIX}_{split}.h5"
    if path.exists():
        with h5py.File(path, "r") as f:
            img_model = f.attrs.get("image_model", "unknown")
            print(f"{split}: caption={f['caption_features'].shape}, image={f['image_features'].shape}, "
                  f"model={img_model}, articles={len(set(f['article_ids'][:]))}")
    else:
        print(f"{split}: {path.name} not found")

print(f"\nReady for Step 2 (training with AdaBelief).")
print(f"Use FILE_PREFIX='{FILE_PREFIX}' in training notebooks.")

train: caption=(3537, 128, 768), image=(3537, 1024), model=clip-vit-L-14, articles=1653
dev: caption=(1054, 128, 768), image=(1054, 1024), model=clip-vit-L-14, articles=519
test: caption=(876, 128, 768), image=(876, 1024), model=clip-vit-L-14, articles=412

Ready for Step 2 (training with AdaBelief).
Use FILE_PREFIX='coolant_clip_vit_L_14_visobert' in training notebooks.
